In [1]:
import os,re, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Sequential, optimizers
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from collections import Counter
from sklearn.model_selection import KFold, train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from itertools import cycle
from tensorflow.keras.layers import Layer

gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:98: UserWarning: unable to load libtensorflow_io_plugins.so: unable to open file: libtensorflow_io_plugins.so, from paths: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so']
caused by: ['/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/libtensorflow_io_plugins.so: undefined symbol: _ZN3tsl6StatusC1EN10tensorflow5error4CodeESt17basic_string_viewIcSt11char_traitsIcEENS_14SourceLocationE']
  warnings.warn(f"unable to load libtensorflow_io_plugins.so: {e}")
/opt/conda/lib/python3.10/site-packages/tensorflow_io/python/ops/__init__.py:104: UserWarning: file system plugins are not loaded: unable to open file: l

In [3]:
def specific(confusion_matrix):
    tn, fp, fn, tp = confusion_matrix.ravel()
    return tn / (fp + tn)
    

def save_predict_result(data, output):
    with open(output, 'w') as f:
        if len(data)>1:
            for i in range(len(data)):
                f.write('# result for fold %d\n' % (i + 1))
                for j in range(len(data[i])):
                    f.write('%d\t%s\n' % (data[i][j][0], data[i][j][2]))
        else:
            for i in range(len(data)):
                f.write('# result for predict\n')
                for j in range(len(data[i])):
                    f.write('%d\t%s\n' % (data[i][j][0], data[i][j][2]))
        f.close()
    return None
    
# Plot the ROC curve and return the AUC value
def plot_roc_curve(data, output, label_column=0, score_column=2):
    datasize = len(data)
    tprs = []
    aucs = []
    fprArray = []
    tprArray = []
    thresholdsArray = []
    mean_fpr = np.linspace(0, 1, 100)
    for i in range(len(data)):
        fpr, tpr, thresholds = roc_curve(data[i][:, label_column], data[i][:, score_column])
        fprArray.append(fpr)
        tprArray.append(tpr)
        thresholdsArray.append(thresholds)
        tprs.append(np.interp(mean_fpr, fpr, tpr))
        tprs[-1][0] = 0.0
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
    colors = cycle(['aqua', 'darkorange', 'cornflowerblue', 'blueviolet', 'deeppink'])
    fig = plt.figure(figsize=(7,7), dpi=300)
    for i, color in zip(range(len(fprArray)), colors):
        if datasize>1:
            plt.plot(fprArray[i], tprArray[i], lw=1, alpha=0.7, color=color,
                     label='ROC fold %d (AUC = %0.3f)' % (i + 1, aucs[i]))
        else:
            plt.plot(fprArray[i], tprArray[i], lw=1, alpha=0.7, color=color,
                     label='ROC (AUC = %0.3f)' %aucs[i])
    plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r',
             label='Random', alpha=.8)
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = auc(mean_fpr, mean_tpr)
    # Calculate the standard deviation
    std_auc = np.std(aucs)
    if datasize>1:
        plt.plot(mean_fpr, mean_tpr, color='blue',
                 label=r'Mean ROC (AUC = %0.4f $\pm$ %0.3f)' % (mean_auc, std_auc),
                 lw=2, alpha=.9)
    std_tpr = np.std(tprs, axis=0)
    tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
    if datasize>1:
        plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='grey', alpha=.2,
                         label=r'$\pm$ 1 std. dev.')
    plt.xlim([0, 1.0])
    plt.ylim([0, 1.0])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc="lower right")
    plt.savefig(output, dpi=800, format='svg')
    plt.close(0)
    return mean_auc, aucs
   
    
# Calculate and save performance metrics
def calculate_metrics(labels, scores, cutoff=0.5, po_label=1):
    my_metrics = {
        'SN': 'NA',
        'SP': 'NA',
        'ACC': 'NA',
        'MCC': 'NA',
        'Recall': 'NA',
        'Precision': 'NA',
        'F1-score': 'NA',
        'Cutoff': cutoff,
    }

    tp, tn, fp, fn = 0, 0, 0, 0
    for i in range(len(scores)):
        if labels[i] == po_label:
            if scores[i] >= cutoff:
                tp = tp + 1
            else:
                fn = fn + 1
        else:
            if scores[i] < cutoff:
                tn = tn + 1
            else:
                fp = fp + 1

    my_metrics['SN'] = tp / (tp + fn) if (tp + fn) != 0 else 0
    my_metrics['SP'] = tn / (fp + tn) if (fp + tn) != 0 else 0
    my_metrics['ACC'] = (tp + tn) / (tp + fn + tn + fp)
    my_metrics['MCC'] = (tp * tn - fp * fn) / np.math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) if (
                                                                                                                     tp + fp) * (
                                                                                                                     tp + fn) * (
                                                                                                                     tn + fp) * (
                                                                                                                     tn + fn) != 0 else 0
    my_metrics['Precision'] = tp / (tp + fp) if (tp + fp) != 0 else 0
    my_metrics['Recall'] = my_metrics['SN']
    my_metrics['F1-score'] = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) != 0 else 0
    return my_metrics

def calculate_metrics_list(data, label_column=0, score_column=2, cutoff=0.5, po_label=1):
    metrics_list = []
    for i in data:
        metrics_list.append(calculate_metrics(i[:, label_column], i[:, score_column], cutoff=cutoff, po_label=po_label))
    if len(metrics_list) == 1:
        return metrics_list
    else:
        mean_dict = {}
        std_dict = {}
        keys = metrics_list[0].keys()
        for i in keys:
            mean_list = []
            for metric in metrics_list:
                mean_list.append(metric[i])
            mean_dict[i] = np.array(mean_list, dtype=np.float32).sum() / len(metrics_list)
            std_dict[i] = np.array(mean_list, dtype=np.float32).std()
        metrics_list.append(mean_dict)
        metrics_list.append(std_dict)
        return metrics_list
        
def save_prediction_metrics_list(metrics_list, output):
    if len(metrics_list) == 1:
        with open(output, 'w') as f:
            f.write('Result')
            for keys in metrics_list[0]:
                f.write('\t%s' % keys)
            f.write('\n')
            for i in range(len(metrics_list)):
                f.write('value')
                for keys in metrics_list[i]:
                    f.write('\t%s' % metrics_list[i][keys])
                f.write('\n')
            f.close()
    else:
        with open(output, 'w') as f:
            f.write('Fold')
            for keys in metrics_list[0]:
                f.write('\t%s' % keys)
            f.write('\n')
            for i in range(len(metrics_list)):
                if i <= len(metrics_list)-3:
                    f.write('%d' % (i + 1))
                elif i == len(metrics_list)-2:
                    f.write('mean')
                else:
                    f.write('std')
                for keys in metrics_list[i]:
                    f.write('\t%s' % metrics_list[i][keys])
                f.write('\n')
            f.close()
    return None       
        
        
def save_result(cv_res, ind_res, outPath, codename, cutoff):
    out = os.path.join(outPath, codename.lower())
    save_predict_result(cv_res, out + '_pre_cv.txt')
    cv_meanauc, cv_auc = plot_roc_curve(cv_res, out + '_roc_cv.svg', label_column=0, score_column=2)
    cv_metrics = calculate_metrics_list(cv_res, label_column=0, score_column=2, cutoff=cutoff, po_label=1)
    save_prediction_metrics_list(cv_metrics, out + '_metrics_cv.txt')
    save_predict_result(ind_res, out + '_pre_ind.txt')
    ind_meanauc, ind_auc = plot_roc_curve(ind_res, out + '_roc_ind.svg', label_column=0, score_column=2)
    ind_metrics = calculate_metrics_list(ind_res, label_column=0, score_column=2, cutoff=cutoff, po_label=1)
    save_prediction_metrics_list(ind_metrics, out + '_metrics_ind.txt')
    return ind_metrics


def save_val_result(cv_res, outPath, codename, cutoff):
    out = os.path.join(outPath, codename.lower())
    save_predict_result(cv_res, out + '_pre_cv.txt')
    cv_meanauc, cv_auc = plot_roc_curve(cv_res, out + '_roc_cv.svg', label_column=0, score_column=2)
    cv_metrics = calculate_metrics_list(cv_res, label_column=0, score_column=2, cutoff=cutoff, po_label=1)
    save_prediction_metrics_list(cv_metrics, out + '_metrics_cv.txt')
    return cv_metrics


# Create folder
def mkdir(path):
    path=path.strip()
    path=path.rstrip("\\")
    # Check if the path exists
    isExists=os.path.exists(path)
    if not isExists:
        # Create the directory if it doesn't exist
        os.makedirs(path)
    else:
        # Do not create directory if it exists
        pass

In [4]:
# The code of ProteinBert Encoding.
ALL_AAS = 'ACDEFGHIKLMNPQRSTUVWXY'
ADDITIONAL_TOKENS = ['<OTHER>', '<START>', '<END>', '<PAD>']

# Each sequence is added <START> and <END> tokens
ADDED_TOKENS_PER_SEQ = 2

n_aas = len(ALL_AAS)
aa_to_token_index = {aa: i for i, aa in enumerate(ALL_AAS)}
additional_token_to_index = {token: i + n_aas for i, token in enumerate(ADDITIONAL_TOKENS)}
token_to_index = {**aa_to_token_index, **additional_token_to_index}
index_to_token = {index: token for token, index in token_to_index.items()}
n_tokens = len(token_to_index)


def tokenize_seq(seq):
    other_token_index = additional_token_to_index['<OTHER>']
    return [aa_to_token_index.get(aa, other_token_index) for aa in parse_seq(seq)]


def parse_seq(seq):
    if isinstance(seq, str):
        return seq
    elif isinstance(seq, bytes):
        return seq.decode('utf8')
    else:
        raise TypeError('Unexpected sequence type: %s' % type(seq))


def tokenize_seqs(seqs):
    # Note that tokenize_seq already adds <START> and <END> tokens.
    seqs_list = []
    for seq_tokens in map(tokenize_seq, seqs):
        seqs_list.append(seq_tokens)
    return seqs_list


def extract_embedding_features(seqs):
    seqs_list = tokenize_seqs(seqs)
    model = tf.keras.models.load_model('./models/protein_bert')
    embedding_layer = model.layers[3]
    narrow_conv_block1_layer = model.layers[6]
    wide_conv_block1_layer = model.layers[7]
    emb_feature = embedding_layer(np.array(seqs_list))
    narrow_feature = narrow_conv_block1_layer(emb_feature)
    wide_feature = wide_conv_block1_layer(emb_feature)
    feature = tf.add(narrow_feature, wide_feature)
    return np.float32(feature.numpy())

In [5]:
# The code of Integer Encoding
def to_embedding_numeric(seqs):
    numeric_sequences = []
    base_to_index = {'A': 0, 'C': 1, 'D': 2, 'E': 3,
                     'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 'M': 10,
                     'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19, 'X': 20
                     }
    for sequence in seqs:
        sequence = re.sub('[^ACDEFGHIKLMNPQRSTVWYX]', 'X', ''.join(sequence).upper())
        numeric_sequence = []
        for base in sequence:
            numeric_sequence.append(base_to_index[base])
        numeric_sequences.append(numeric_sequence)
    return numeric_sequences

In [6]:
trainfilepath = r'./datasets/train_dataset.csv'
testfilepath = r'./datasets/test_dataset.csv'

In [8]:
train_df = pd.read_csv(trainfilepath)
test_df = pd.read_csv(testfilepath)

In [9]:
y = list(train_df['Label'])
y = np.array(y).astype(np.float32)
y_test = list(test_df['Label'])
y_test = np.array(y_test).astype(np.float32)

In [10]:
protein_bert_train = extract_embedding_features(train_df['Sequence'].values.tolist())
tf.keras.backend.clear_session()
protein_bert_test = extract_embedding_features(test_df['Sequence'].values.tolist())
tf.keras.backend.clear_session()

In [11]:
one_hot_train = np.array(to_embedding_numeric(train_df['Sequence'].values.tolist())).astype(np.float32)
one_hot_test = np.array(to_embedding_numeric(test_df['Sequence'].values.tolist())).astype(np.float32)

In [13]:
# The dataset was processed following the code in the MVNN-HNHC paper to ensure a fair comparison.
def split_train_set(x, y):
    index = np.arange(y.shape[0])
    np.random.shuffle(index)
    x = x[index]
    y = y[index]
    train_pos_idx = y == 1.0
    train_neg_idx = y == 0.0
    x_pos = x[train_pos_idx][:12262]
    x_neg = x[train_neg_idx][:12262]
    y_pos = [1 for i in range(12262)]
    y_neg = [0 for i in range(12262)]
    x_new = np.concatenate((x_pos, x_neg), axis=0)
    y_new = y_pos + y_neg
    return x_new, np.array(y_new).astype(np.float32)

def split_test_set(x, y):
    index = np.arange(y.shape[0])
    np.random.shuffle(index)
    x = x[index]
    y = y[index]
    train_pos_idx = y == 1.0
    train_neg_idx = y == 0.0
    x_pos = x[train_pos_idx][:3343]
    x_neg = x[train_neg_idx][:3343]
    y_pos = [1 for i in range(3343)]
    y_neg = [0 for i in range(3343)]
    x_new = np.concatenate((x_pos, x_neg), axis=0)
    y_new = y_pos + y_neg
    return x_new, np.array(y_new).astype(np.float32)

In [14]:
protein_bert_train, y_1 = split_train_set(protein_bert_train, y)
one_hot_train, y = split_train_set(one_hot_train, y)

protein_bert_test, y_test_1 = split_test_set(protein_bert_test, y_test)
one_hot_test, y_test = split_test_set(one_hot_test, y_test)

In [31]:
def res_net_block(input_data, filters, strides=1):
    x = layers.Conv1D(filters, kernel_size=3, strides=strides, padding='same')(input_data)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Conv1D(filters, kernel_size=3, strides=1, padding='same')(x)
    if strides != 1:
        downsample = layers.Conv1D(filters, kernel_size=1, strides=strides)(input_data)
    else:  
        downsample = input_data
    x = layers.Add()([x, downsample])
    x = layers.BatchNormalization()(x)
    output = layers.Activation('relu')(x)
    return output


def concat_layer(intputs):
    return tf.concat(intputs, axis=1)


def multiply_layer(inputs):
    x, y = inputs
    return tf.multiply(x, y)

def ResNet(encode):
    inputs = tf.keras.Input(shape=(encode.shape[1], encode.shape[2]))
    x = layers.Conv1D(64, kernel_size=3)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPool1D(pool_size=2, strides=1, padding='same')(x)
    x = layers.Dropout(0.4)(x)

    x = res_net_block(x, 64)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dropout(0.4)(x)

    x = res_net_block(x, 64)
    x = layers.MaxPool1D(2)(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Flatten()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(128, activation=tf.nn.gelu)(x)
    x = layers.Dropout(0.4)(x)
    x_shortcut1 = layers.Dense(64, activation=tf.nn.gelu)(x)
    x = layers.Dense(32, activation=tf.nn.gelu)(x_shortcut1)
    output = layers.Dense(1, activation='sigmoid')(x)  
    
    model = Model(inputs=[inputs], outputs=output)
    model.compile(optimizer=optimizers.AdamW(),
                     loss=tf.keras.losses.BinaryCrossentropy(),
                     metrics=['acc'],
                     experimental_run_tf_function=False)
    return model

In [26]:
def build_light_transformer(
    vocab_size=21,          
    max_len=29,             
    embed_dim=64,           
    num_heads=2,            
    ff_dim=128,             
    num_transformer_blocks=1,
    dropout_rate=0.7
):
    inputs = layers.Input(shape=(max_len,))

    embedding_layer = layers.Embedding(vocab_size, embed_dim, input_length=max_len)
    x = embedding_layer(inputs)  # (batch, max_len, embed_dim)

    positions = tf.range(start=0, limit=max_len, delta=1)
    position_embedding = layers.Embedding(input_dim=max_len, output_dim=embed_dim)(positions)
    x = x + position_embedding
    x = layers.Dropout(dropout_rate)(x)

    # 4. Transformer Blocks
    for _ in range(num_transformer_blocks):
        # Multi-Head Self-Attention
        attn_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim // num_heads, dropout=dropout_rate
        )(x, x)
        attn_output = layers.Dropout(dropout_rate)(attn_output)
        x1 = layers.LayerNormalization()(x + attn_output)

        # Feed Forward
        ffn = layers.Dense(ff_dim, activation='relu')(x1)
        ffn = layers.Dense(embed_dim)(ffn)
        ffn = layers.Dropout(dropout_rate)(ffn)
        x = layers.LayerNormalization()(x1 + ffn)

    x = layers.Flatten()(x)  # (batch, embed_dim)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    output = layers.Dense(1, activation='sigmoid')(x)


    model = Model(inputs=inputs, outputs=output, name="LightTransformer")
    
    optimizer = optimizers.AdamW()
    model.compile(
        optimizer=optimizer,
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=['acc']
    )
    return model

In [27]:
def build_attention_integration():
    inputs = layers.Input(shape=(2,), name='model_outputs')

    x = layers.Dense(2, use_bias=False, activation=None)(inputs)
    
    attention_scores = layers.Activation('softmax')(x)

    attention_output = layers.Multiply(name='weighted_outputs')([inputs, attention_scores])

    output = layers.Lambda(lambda x: K.sum(x, axis=1, keepdims=True), name='final_output')(attention_output)

    model = Model(inputs=inputs, outputs=output)

    model.compile(optimizer=optimizers.AdamW(),
                  loss=tf.keras.losses.BinaryCrossentropy(),
                  metrics=['acc'])
    
    return model

## 10-fold cross validation

In [44]:
def train_model(model, model_name, x_train, y, x_valid, y_valid, outputs, batch_size):
    filepath = os.path.join(outputs, model_name)
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_acc', patience=30)
    best_saving = tf.keras.callbacks.ModelCheckpoint(filepath, monitor='val_acc', mode='auto',
                                                       verbose=0, save_best_only=True, save_weights_only=True)
    model.fit(x_train, y, validation_data=(x_valid, y_valid), epochs=200, batch_size=batch_size, shuffle=True,
              callbacks=[early_stopping, best_saving], verbose=0)
    model.load_weights(filepath)
    p = model.predict(x_valid, verbose=0)
    tmp_result = np.zeros((len(y_valid), 3))
    tmp_result[:, 0], tmp_result[:, 1:] = y_valid, p
    return tmp_result, p

In [45]:
from tqdm import tqdm
def cross_validation():
    outputs = r'./Results/'
    mkdir(outputs)
    folds = StratifiedKFold(10, shuffle=True, random_state=42).split(protein_bert_train, y)
    prediction_result_cv = []
    prediction_result_cv_pb = []
    prediction_result_cv_oh = []
    for i, (train, valid) in tqdm(enumerate(folds)):
        train_x_emb, train_y = protein_bert_train[train], y[train]
        valid_x_emb, valid_y = protein_bert_train[valid], y[valid]
        train_x_onehot, valid_x_onehot = one_hot_train[train], one_hot_train[valid]
        modelName = 'model1_' + str(i + 1) + '.h5'
        network_1 = ResNet(train_x_emb)
        tmp_result, p_1 = train_model(network_1, modelName, train_x_emb, train_y, valid_x_emb, valid_y, outputs, 32)
        prediction_result_cv_pb.append(tmp_result)

        modelName = 'model2_' + str(i + 1) + '.h5'
        network_2 = build_light_transformer()
        tmp_result, p_2 = train_model(network_2, modelName, train_x_onehot, train_y, valid_x_onehot, valid_y, outputs, 512)
        prediction_result_cv_oh.append(tmp_result)
        
        att_model_1 = build_attention_integration()
        att_train_input = np.concatenate((network_1.predict(train_x_emb, verbose=0), network_2.predict(train_x_onehot, verbose=0)), axis=-1)
        att_valid_input = np.concatenate((network_1.predict(valid_x_emb, verbose=0), network_2.predict(valid_x_onehot, verbose=0)), axis=-1)
        modelName = 'model_attn_' + str(i + 1) + '.h5'

        filepath_att = os.path.join(outputs, modelName)
        tmp_result, p_3 = train_model(att_model_1, modelName, att_train_input, train_y, att_valid_input, valid_y, outputs, 32)
        prediction_result_cv.append(tmp_result)
        
    save_val_result(prediction_result_cv, outputs, "att_ensemble", 0.5)
    save_val_result(prediction_result_cv_pb, outputs, "protein_bert", 0.5)
    save_val_result(prediction_result_cv_oh, outputs, "one_hot",0.5)

In [ ]:
flag = cross_validation()

# independent test

In [47]:
def train_model(model, model_name, x_train, train_y, x_valid, y_valid, outputs, batch_size):
    filepath = os.path.join(outputs, model_name)
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_acc', patience=30)
    best_saving = tf.keras.callbacks.ModelCheckpoint(filepath, monitor='val_acc', mode='auto',
                                                       verbose=0, save_best_only=True, save_weights_only=True)
    model.fit(x_train, train_y, validation_data=(x_valid, y_valid), epochs=200, batch_size=batch_size, shuffle=True,
              callbacks=[early_stopping, best_saving], verbose=0)
    model.load_weights(filepath)
    return model

In [48]:
folds = StratifiedKFold(20, shuffle=True, random_state=42).split(protein_bert_train, y)
train_index, valid_index = next(folds)
train_x_emb, train_y = protein_bert_train[train_index], y[train_index]
valid_x_emb, valid_y = protein_bert_train[valid_index], y[valid_index]
train_x_onehot, valid_x_onehot = one_hot_train[train_index], one_hot_train[valid_index]

outputs = r'./Results/'
mkdir(outputs)
modelName = 'model1.h5'
model_1 = ResNet(train_x_emb)
model_1 = train_model(model_1, modelName, train_x_emb, train_y, valid_x_emb, valid_y, outputs, 32)

modelName = 'model2.h5'
model_2 = build_light_transformer()
model_2 = train_model(model_2, modelName, train_x_onehot, train_y, valid_x_onehot, valid_y, outputs, 512)

att_model = build_attention_integration()
att_train_input = np.concatenate((model_1.predict(train_x_emb, verbose=0),
                                  model_2.predict(train_x_onehot, verbose=0)), axis=-1)
att_valid_input = np.concatenate((model_1.predict(valid_x_emb, verbose=0),
                                  model_2.predict(valid_x_onehot, verbose=0)), axis=-1)
modelName = 'model_ensemble.h5'
filepath_att = os.path.join(outputs, modelName)
att_model = train_model(att_model, modelName, att_train_input, train_y, att_valid_input, valid_y, outputs, 32)

In [49]:
ensemble_test_input = np.concatenate((model_1.predict(protein_bert_test, verbose=0),
                                  model_2.predict(one_hot_test, verbose=0)), axis=-1)

In [ ]:
prediction_result_ind = []
p = att_model.predict(ensemble_test_input, verbose=0)
tmp_result = np.zeros((len(y_test), 3))
tmp_result[:, 0], tmp_result[:, 1:] = y_test, p
prediction_result_ind.append(tmp_result)
save_val_result(prediction_result_ind, outputs, "Independent_test", 0.5)

# machine learning

In [ ]:
X = protein_bert_train.reshape(protein_bert_train.shape[0], -1)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import recall_score, confusion_matrix, accuracy_score, precision_score, matthews_corrcoef

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

def evaluate_model(y_true, y_pred):
    sn = recall_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    pre = precision_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel().tolist()
    sp = tn / (fp + tn) if (fp + tn) != 0 else 0
    return sn, sp, acc, mcc, pre

models = {
    'SVM': SVC(max_iter=2000),
    'RF': RandomForestClassifier(random_state=42),
    'LR': LogisticRegression(random_state=42, max_iter=1000),
    'DT': DecisionTreeClassifier()
}

model_performance = {}

for model_name, model in models.items():
    print(f"\n===== Training {model_name} Model=====")


    sn_list, sp_list, acc_list, mcc_list, pre_list = [], [], [], [], []
    for train_idx, test_idx in skf.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        sn, sp, acc, mcc, pre = evaluate_model(y_test, y_pred)

        sn_list.append(sn)
        sp_list.append(sp)
        acc_list.append(acc)
        mcc_list.append(mcc)
        pre_list.append(pre)

    model_performance[model_name] = {
        'mean_sn': np.mean(sn_list),
        'mean_sp': np.mean(sp_list),
        'mean_acc': np.mean(acc_list),
        'mean_mcc': np.mean(mcc_list),
        'mean_pre': np.mean(pre_list)
    }

    print(f"{model_name} 10-fold cross-validation performance：Sn={np.mean(sn_list):.4f}, "
          f"Sp={np.mean(sp_list):.4f}, "
          f"Acc={np.mean(acc_list):.4f}, "
          f"MCC={np.mean(mcc_list):.4f}, "
          f"Pre={np.mean(pre_list):.4f}")

pd.DataFrame(model_performance).T.to_csv("traditional_ml_performance.csv", index=True, encoding='utf-8-sig')

# Finetune ESM-2

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

In [4]:
# -*- coding: utf-8 -*-
"""
ESM-2-650M fine-tuning for Kcr site prediction.
Single-GPU Mode + 10-fold CV + Mixed Precision (AMP) + Last 2 Layers Tuning.
"""

import pandas as pd
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score, matthews_corrcoef, roc_auc_score,
    confusion_matrix, roc_curve
)
from sklearn.model_selection import KFold
from tqdm import tqdm

# ======================== Configuration ========================
class Config:
    MODEL_NAME = "facebook/esm2_t30_150M_UR50D"
    MAX_LEN = 29
    NUM_LABELS = 1
    BATCH_SIZE = 128
    EPOCHS = 20
    LR = 2e-5
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.1
    N_FOLDS = 10
    SEED = 42
    OUTPUT_DIR = "./esm2_kcr_output"
    FP16 = True

cfg = Config()

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

torch.manual_seed(cfg.SEED)
torch.cuda.manual_seed_all(cfg.SEED)
np.random.seed(cfg.SEED)


# ======================== Dataset ========================
class KcrDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len=33):
        self.sequences = list(sequences)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = str(self.sequences[idx])
        label = float(self.labels[idx])
        encoding = self.tokenizer(
            seq, max_length=self.max_len + 2,
            padding="max_length", truncation=True, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(label, dtype=torch.float)
        }


# ======================== Model ========================
class ESM2ForKcr(nn.Module):
    def __init__(self, model_name, num_labels=1, dropout=0.1):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.esm = AutoModel.from_pretrained(model_name)

        for param in self.esm.parameters():
            param.requires_grad = False

        if hasattr(self.esm, "encoder") and hasattr(self.esm.encoder, "layer"):
            for layer in self.esm.encoder.layer[-2:]:
                for param in layer.parameters():
                    param.requires_grad = True
            print("--> Successfully unfrozen the last 2 layers of ESM-2.")
        else:
            print("--> Warning: Could not find encoder layers. Standard freezing applied.")

        hidden_size = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.esm(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (last_hidden * mask).sum(1) / mask.sum(1)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)

        loss = None
        if labels is not None:
            loss = nn.BCEWithLogitsLoss()(logits, labels)
        return {"loss": loss, "logits": logits}


# ======================== Metrics ========================
def compute_metrics(labels, probs, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sn = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    sp = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    acc = accuracy_score(labels, preds)
    mcc = matthews_corrcoef(labels, preds)
    auc = roc_auc_score(labels, probs)
    return {"Sn": sn, "Sp": sp, "Precision": precision, "Acc": acc, "MCC": mcc, "AUC": auc}



# ======================== Train / Eval ========================
def train_one_epoch(model, dataloader, optimizer, scheduler, scaler, device):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(dataloader, desc="Training", leave=False)

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()

        if cfg.FP16 and scaler is not None:
            with autocast(enabled=True):  # 兼容旧版本，不加 device_type
                outputs = model(input_ids, attention_mask, labels)
                loss = outputs["loss"]

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            # 兼容旧版的学习率安全控制
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            scale_after = scaler.get_scale()
            if scale_before <= scale_after:
                scheduler.step()
        else:
            outputs = model(input_ids, attention_mask, labels)
            loss = outputs["loss"]
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    all_probs, all_labels = [], []

    progress_bar = tqdm(dataloader, desc="Evaluating", leave=False)

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        if cfg.FP16:
            with autocast(enabled=True):
                outputs = model(input_ids, attention_mask)
        else:
            outputs = model(input_ids, attention_mask)

        probs = torch.sigmoid(outputs["logits"])
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    fpr, tpr, thresholds = roc_curve(all_labels, all_probs)
    best_threshold = thresholds[np.argmax(tpr - fpr)]
    metrics = compute_metrics(all_labels, all_probs, threshold=best_threshold)
    metrics["threshold"] = best_threshold

    return metrics, all_probs, all_labels


# ======================== 10-Fold CV ========================
def run_10fold_cv(sequences, labels):
    tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME)
    sequences = np.array(sequences)
    labels = np.array(labels)

    kf = KFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.SEED)
    fold_results = []
    all_fold_probs = []
    all_fold_labels = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(sequences), 1):
        print(f"\n{'#'*60}\n# Fold {fold}/{cfg.N_FOLDS}\n{'#'*60}")

        train_seqs, val_seqs = sequences[train_idx], sequences[val_idx]
        train_labs, val_labs = labels[train_idx], labels[val_idx]

        train_dataset = KcrDataset(train_seqs.tolist(), train_labs.tolist(), tokenizer, cfg.MAX_LEN)
        val_dataset = KcrDataset(val_seqs.tolist(), val_labs.tolist(), tokenizer, cfg.MAX_LEN)

        train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

        model = ESM2ForKcr(cfg.MODEL_NAME, num_labels=cfg.NUM_LABELS)
        model.to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
        total_steps = len(train_loader) * cfg.EPOCHS
        warmup_steps = int(total_steps * cfg.WARMUP_RATIO)
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

        scaler = GradScaler() if cfg.FP16 else None

        best_mcc = -1.0
        best_metrics = None
        best_probs = None
        best_labels = None

        for epoch in range(1, cfg.EPOCHS + 1):
            train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, DEVICE)
            metrics, probs, labs = evaluate(model, val_loader, DEVICE)

            print(f"  Epoch {epoch:2d} | loss: {train_loss:.4f} | "
                  f"Sn: {metrics['Sn']:.4f}  Acc: {metrics['Acc']:.4f}  "
                  f"MCC: {metrics['MCC']:.4f}  AUC: {metrics['AUC']:.4f}")

            if metrics["MCC"] > best_mcc:
                best_mcc = metrics["MCC"]
                best_metrics = metrics
                best_probs = probs
                best_labels = labs

        torch.save(model.state_dict(), os.path.join(cfg.OUTPUT_DIR, f"fold{fold}_best_model.pt"))

        fold_results.append(best_metrics)
        all_fold_probs.extend(best_probs)
        all_fold_labels.extend(best_labels)

        print(f"\n  Fold {fold} Best -> Sn: {best_metrics['Sn']:.4f}  Sp: {best_metrics['Sp']:.4f}  "
              f"Acc: {best_metrics['Acc']:.4f}  MCC: {best_metrics['MCC']:.4f}  AUC: {best_metrics['AUC']:.4f}")

        del model
        torch.cuda.empty_cache()

    return fold_results, np.array(all_fold_probs), np.array(all_fold_labels)


# ======================== Summary ========================
def summarize_results(fold_results, all_probs, all_labels):
    print(f"\n{'='*70}\n10-Fold Cross-Validation Results\n{'='*70}")
    df = pd.DataFrame(fold_results)
    df.index = [f"Fold {i+1}" for i in range(len(df))]
    print("\nPer-fold results:")
    print(df[['Sn', 'Sp', 'Precision', 'Acc', 'MCC', 'AUC']].to_string(float_format="%.4f"))

    print(f"\n{'='*70}\nAverage (mean ± std):")
    for metric in ['Sn', 'Sp', 'Precision', 'Acc', 'MCC', 'AUC']:
        print(f"  {metric:9s}: {df[metric].mean():.4f} ± {df[metric].std():.4f}")


    df.to_csv(os.path.join(cfg.OUTPUT_DIR, "per_fold_results.csv"))
    summary = {m + '_mean': df[m].mean() for m in ['Sn', 'Sp', 'Precision', 'Acc', 'MCC', 'AUC']}
    summary.update({m + '_std': df[m].std() for m in ['Sn', 'Sp', 'Precision', 'Acc', 'MCC', 'AUC']})
    pd.DataFrame([summary]).to_csv(os.path.join(cfg.OUTPUT_DIR, "summary_results.csv"), index=False)
    print(f"\nResults saved to {cfg.OUTPUT_DIR}/")


# ======================== Main ========================
def main(sequences, labels):
    print(f"Total samples: {len(sequences)}")
    print(f"Positive: {sum(labels)}  Negative: {len(labels) - sum(labels)}")
    print(f"Model: {cfg.MODEL_NAME}")
    print(f"Batch Size: {cfg.BATCH_SIZE} | FP16 AMP: {cfg.FP16}")

    fold_results, all_probs, all_labels = run_10fold_cv(sequences, labels)
    summarize_results(fold_results, all_probs, all_labels)

Using device: cuda


In [5]:
def split_train_set(x, y):
    train_pos_idx = y == 1.0
    train_neg_idx = y == 0.0
    x_pos = x[train_pos_idx][:12262]
    x_neg = x[train_neg_idx][:12262]
    y_pos = [1.0 for i in range(12262)]
    y_neg = [0.0 for i in range(12262)]
    x_new = np.concatenate((x_pos, x_neg), axis=0)
    y_new = y_pos + y_neg
    return x_new, np.array(y_new).astype(np.float32)

In [6]:
df = pd.read_csv("./datasets/train_dataset.csv")
sequences, labels = split_train_set(df['Sequence'], df['Label'])

In [ ]:
main(sequences, labels)